# Learned-LIF Connectivity Inference (Modular)

This notebook is the import-based entry point for the learned-LIF inference package. It keeps the spike-only and voltage-augmented connectivity pipelines in reusable Python modules under `lif_inference/` and uses gated execution cells so you can run only the inference path you want.

**Before you run anything:**
- Make sure you already have a saved session under `LIF data/<timestamp>` with `recordingXXX.npz` files and a matching `network_*.npz` file.
- Set `session_source` to `latest` or to a specific saved session folder.
- Leave both execution toggles off until the session-resolution cell prints the session you expect.
- The voltage-augmented path needs saved voltage traces; the current conductance simulation pipeline writes raw full-dt voltage by default.

**Suggested first run order:**
1. Run the import cell.
2. Review the parameter cell and keep both execution toggles off.
3. Run the session-resolution cell and confirm the resolved session path.
4. Turn on `execute_spike_only` first if you want the simpler spike-only baseline.
5. Turn on `execute_voltage_augmented` only when you want masked-voltage supervision and the saved session includes voltage traces.
6. Keep `use_all_recordings = True` unless you are deliberately debugging on a smaller subset.

**What to expect:**
- Spike-only inference gives the simpler baseline and is usually the easier first check.
- Voltage-augmented inference adds masked subthreshold voltage supervision and usually takes more setup care.
- Both paths save learned connectivity artifacts through the packaged `lif_inference/` workflow rather than notebook-local helper code.

In [2]:
from pathlib import Path
from pprint import pprint

from lif_inference import (
    run_learned_lif_pipeline,
    run_voltage_augmented_pipeline,
)

In [11]:
session_source = "latest"

execute_spike_only = False
execute_voltage_augmented = False

spike_only_params = {
    "K": 50,
    "recording_idx": 0,
    "n_epochs": 40,
    "lr": 1e-3,
    "batch_size": 128,
    "patience": 20,
    "dt": 1.0,
    "max_delay": 8,
    "l1_lambda": 0.01,
    "pos_weight": 5.0,
    "val_fraction": 0.2,
    "device": "cpu",
    "output_tag": "notebook",
    "subsample_T": None,
    "use_all_recordings": True,
    "candidate_mode": "hybrid",
    "candidate_spatial_frac": 0.8,
    "candidate_min_lag": 1,
    "candidate_max_lag": None,
    "exclude_detected_bursts": False,
}

voltage_augmented_params = {
    "K": 50,
    "recording_idx": 0,
    "n_epochs": 40,
    "lr": 1e-3,
    "batch_size": 128,
    "patience": 20,
    "dt": 1.0,
    "max_delay": 8,
    "l1_lambda": 0.01,
    "pos_weight": 5.0,
    "voltage_lambda": 1.0,
    "val_fraction": 0.2,
    "device": "cpu",
    "output_tag": "notebook",
    "subsample_T": None,
    "use_all_recordings": True,
    "candidate_mode": "hybrid",
    "candidate_spatial_frac": 0.8,
    "candidate_min_lag": 1,
    "candidate_max_lag": None,
    "mask_pre_ms": 1.0,
    "mask_post_ms": 5.0,
    "peak_threshold_mv": 15.0,
}

In [ ]:
def resolve_session_dir(session_source, data_root=Path("LIF data")):
    source_path = Path(session_source)
    if source_path.exists():
        return source_path.resolve()

    candidate = (data_root / session_source).resolve()
    if session_source != "latest" and candidate.exists():
        return candidate

    sessions = sorted(
        (
            path for path in data_root.rglob("*")
            if path.is_dir()
            and any(path.glob("recording[0-9][0-9][0-9].npz"))
            and any(path.glob("network_*.npz"))
        ),
        key=lambda path: path.as_posix(),
    )
    if not sessions:
        raise FileNotFoundError(f"No saved sessions found under {data_root}")
    if session_source == "latest":
        return sessions[-1].resolve()
    raise FileNotFoundError(f"Session source not found: {session_source}")


session_dir = resolve_session_dir(session_source)
print(f"Resolved session: {session_dir.name}")
print(f"Session path: {session_dir}")

In [ ]:
if execute_spike_only:
    spike_only_results, spike_only_conn_matrix = run_learned_lif_pipeline(
        str(session_dir),
        **spike_only_params,
    )
    print("Spike-only learned-LIF finished.")
    pprint({
        "auc": spike_only_results.get("auc"),
        "ap": spike_only_results.get("ap"),
        "f1": spike_only_results.get("f1"),
        "threshold": spike_only_results.get("threshold"),
    })
else:
    spike_only_results = None
    spike_only_conn_matrix = None
    print("Spike-only inference skipped. Set execute_spike_only = True to run it.")

In [ ]:
# CCG (cross-correlogram) baseline: training-free, scored on the SAME inputs the
# spike-only learned model used (surfaced in spike_only_results). Run cell 4 first
# (execute_spike_only = True) so spike_only_results is populated.
from lif_inference.ccg_baseline import run_ccg_baseline

if spike_only_results is not None:
    r = spike_only_results
    ccg = run_ccg_baseline(
        r['spike_matrix'], r['neighbor_indices'], neuron_ids=r['neuron_ids'],
        true_binary=r['true_binary'], boundaries=r['boundaries'],
        excluded_bins=r['excluded_bins'],
        threshold_mode='oracle_f1',     # oracle ceiling, apples-to-apples with the learned eval
        min_lag=1, max_lag=5,           # 1-5 ms causal window at dt=1.0 ms/bin (matches max_delay=5)
    )
    print('CCG baseline (oracle):', ccg['metrics'])
    print('learned model        :', {k: r.get(k) for k in ('auc', 'ap', 'f1')})
else:
    print('Run the spike-only learned model first (set execute_spike_only = True) to populate spike_only_results.')

In [6]:
if execute_voltage_augmented:
    voltage_augmented_results, voltage_augmented_conn_matrix = run_voltage_augmented_pipeline(
        str(session_dir),
        **voltage_augmented_params,
    )
    print("Voltage-augmented learned-LIF finished.")
    pprint({
        "auc": voltage_augmented_results.get("auc"),
        "ap": voltage_augmented_results.get("ap"),
        "f1": voltage_augmented_results.get("f1"),
        "sign_accuracy": voltage_augmented_results.get("sign_accuracy"),
        "weight_corr": voltage_augmented_results.get("weight_corr"),
    })
else:
    voltage_augmented_results = None
    voltage_augmented_conn_matrix = None
    print("Voltage-augmented inference skipped. Set execute_voltage_augmented = True to run it.")

Voltage-augmented inference skipped. Set execute_voltage_augmented = True to run it.


In [10]:
summary = {}
if spike_only_results is not None:
    summary["spike_only"] = {
        "auc": spike_only_results.get("auc"),
        "ap": spike_only_results.get("ap"),
        "f1": spike_only_results.get("f1"),
    }
if voltage_augmented_results is not None:
    summary["voltage_augmented"] = {
        "auc": voltage_augmented_results.get("auc"),
        "ap": voltage_augmented_results.get("ap"),
        "f1": voltage_augmented_results.get("f1"),
        "sign_accuracy": voltage_augmented_results.get("sign_accuracy"),
        "weight_corr": voltage_augmented_results.get("weight_corr"),
    }

if summary:
    pprint(summary)
else:
    print("No inference runs have been executed yet.")

{'spike_only': {'ap': np.float64(0.1925925925925926),
                'auc': np.float64(0.5),
                'f1': 0.32298136643254505}}
